# NN-09 · Hyper-Connections at small scale — Colab runner

Runs the study's code on **your Colab GPU** and keeps everything on **Google Drive**. Nothing here needs the project repo:
the code is embedded in this notebook (snapshot below).

**How to use**
1. *Runtime ▸ Change runtime type ▸ pick a GPU* (T4 is fine for the smoke plan; faster GPUs help a lot for the real plans).
2. Leave **PLAN = smoke** for the first run and press *Runtime ▸ Run all*. It checks the GPU, runs the test-suite and runs
   every model × connection for 2 tiny epochs (about 20-40 min). This proves the pipeline works on your Colab.
3. For real runs set **PLAN** to `pilot` (a small first study) or `tier1` (the full 144-run matrix) and type your **EPOCHS**
   and **SEEDS**. First **untick LAUNCH** and run: cell 7 times your GPU (about 5 min) and prints the GPU-hours of that plan
   and of other epoch / seed choices. When you are happy, **tick LAUNCH** and run again. Finished runs are skipped and
   interrupted ones resume, so after a disconnect (Colab sessions end after at most ~12 h) just *Run all* again.

**What you get back**
- the executed notebook: *File ▸ Download ▸ .ipynb*, then replace `notebooks/nn09_colab.ipynb` in the project;
- a results zip on Drive (`MyDrive/<DRIVE_FOLDER>/<plan>_raw.zip`): unzip it into the project's `results/raw/`.

**Honest limits**
- `EPOCHS` has no approved value yet: D7/D8 say 200 (provisional) and the P4 plan decides. The default 30 is only a placeholder.
- D6 (how a 10 % subset is trained) is still PENDING; **P10_STEPS = same_steps** is the recommended default, not a decision.
- Training is fp32. Whether to allow mixed precision on Colab GPUs is an open question for the project lead.
- Pilot / tier1 results are real logs of the study's pipeline, but nothing here is a conclusion; the analysis comes later.

In [1]:
#@title 1 · What to run  { display-mode: "form" }
PLAN = "smoke"            #@param ["smoke", "pilot", "tier1"]
EPOCHS = 30               #@param {type:"integer"}
SEEDS = "0"               #@param {type:"string"}
P10_STEPS = "same_steps"  #@param ["same_steps", "same_epochs"]
CKPT_EVERY = 1            #@param {type:"integer"}
RUN_TESTS = True          #@param {type:"boolean"}
RUN_TIMING = True         #@param {type:"boolean"}
LAUNCH = True             #@param {type:"boolean"}
DRIVE_FOLDER = "nn09"     #@param {type:"string"}
# PLAN: smoke = all 24 model x connection pairs for 2 tiny epochs; pilot = ResNet-20/56 + ViT-Tiny-6, 4 connections,
#       CIFAR-100 100% and 10%, SEEDS, EPOCHS; tier1 = the full 6 models x 4 connections x 2 data settings, SEEDS, EPOCHS.
# SEEDS: "0" or "0,1,2".   CKPT_EVERY: resume-checkpoint cadence in epochs (raise to 5 if Drive writes are slow).
# DRIVE_FOLDER: no spaces. One results folder is used per plan and epoch count (e.g. pilot_e30) so plans never mix.

In [2]:
#@title Project code snapshot (generated by scripts/make_colab_notebook.py; do not edit)  { display-mode: "form" }
SNAPSHOT_COMMIT = "db0a1dfdba0bc8b6dbae5654f989608e2798908c"
SNAPSHOT_B64 = "".join([
    "...",
])

In [3]:
# 3 · Setup: GPU check, Google Drive, project code, helper
import base64, glob, io, json, os, pathlib, shutil, subprocess, sys, time, zipfile
import torch

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

assert torch.cuda.is_available(), "No GPU found. In Colab: Runtime > Change runtime type > pick a GPU, then run again."
props = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__} | CUDA {torch.version.cuda} | {props.name} | {props.total_memory / 2**30:.1f} GiB")

if IN_COLAB:
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive") / DRIVE_FOLDER
    WORK = pathlib.Path("/content")
else:                                              # running elsewhere (e.g. a cloud VM): use local folders
    BASE = pathlib.Path(os.environ.get("NN09_OUT", "nn09_out")).resolve()
    WORK = pathlib.Path("nn09_work").resolve()
BASE.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

CODE_DIR = WORK / "nn09_code"
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
zipfile.ZipFile(io.BytesIO(base64.b64decode(SNAPSHOT_B64))).extractall(CODE_DIR)
(CODE_DIR / ".snapshot_commit").write_text(SNAPSHOT_COMMIT, encoding="utf-8")
print("project code:", SNAPSHOT_COMMIT, "->", CODE_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "pyyaml"], check=True)
ENV = dict(os.environ, PYTHONPATH=str(CODE_DIR / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""), PYTHONUNBUFFERED="1")


def sh(args, cwd=CODE_DIR):
    """Run a command, stream its output into the cell, raise if it fails."""
    p = subprocess.Popen([str(a) for a in args], cwd=cwd, env=ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed (exit code {p.returncode}): {' '.join(str(a) for a in args)}")

torch 2.11.0+cu130 | CUDA 13.0 | Tesla T4 | 14.6 GiB
Mounted at /content/drive
project code: db0a1dfdba0bc8b6dbae5654f989608e2798908c -> /content/nn09_code


In [4]:
# 4 · Resolve the plan
seeds = [int(s) for s in SEEDS.replace(",", " ").split()]
PLANS = {
    "smoke": dict(tier="smoke", models=None, connections=None, data=None, seeds=None, epochs=None),
    "pilot": dict(tier="1", models=["resnet20", "resnet56", "vit_tiny_d6"],
                  connections=["residual", "dhc_n2", "dhc_n4", "mhc_n4"],
                  data=["cifar100_p100", "cifar100_p10"], seeds=seeds, epochs=EPOCHS),
    "tier1": dict(tier="1", models=None, connections=None, data=None, seeds=seeds, epochs=EPOCHS),
}
plan = PLANS[PLAN]
RUN_NAME = PLAN if plan["epochs"] is None else f"{PLAN}_e{plan['epochs']}"
RAW, CKPT, DATA_DIR = BASE / RUN_NAME / "raw", BASE / RUN_NAME / "runs", WORK / "data"
for d in (RAW, CKPT, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

SETS = ["--set", f"paths.results_dir={RAW}", "--set", f"paths.runs_dir={CKPT}",
        "--set", f"paths.data_dir={DATA_DIR}", "--set", f"train.checkpoint_every={CKPT_EVERY}"]
if plan["epochs"]:
    SETS += ["--set", f"train.epochs={plan['epochs']}"]
    if P10_STEPS == "same_steps":                      # D6(b): 390 = one pass over 50,000 images at batch 128
        SETS += ["--set", "train.epoch_steps=390"]
GRID = []
for flag, key in (("--models", "models"), ("--connections", "connections"), ("--data", "data")):
    if plan[key]:
        GRID += [flag, *plan[key]]
if plan["seeds"]:
    GRID += ["--seeds", *map(str, plan["seeds"])]

listing = subprocess.run([sys.executable, "scripts/run_matrix.py", "--tier", plan["tier"], *GRID, *SETS, "--dry-run"],
                         cwd=CODE_DIR, env=ENV, capture_output=True, text=True, check=True).stdout.splitlines()[0]
print(f"plan {PLAN!r}: {listing}")
print(f"epochs: {plan['epochs'] or '2 (smoke)'} | seeds: {plan['seeds'] or 'smoke default'} | 10%-data budget: {P10_STEPS}")
print(f"results -> {RAW}\ncheckpoints -> {CKPT}")

plan 'smoke': 24 runs, 24 already complete
epochs: 2 (smoke) | seeds: smoke default | 10%-data budget: same_steps
results -> /content/drive/MyDrive/nn09/smoke/raw
checkpoints -> /content/drive/MyDrive/nn09/smoke/runs


In [5]:
# 5 · Test-suite (about 3-5 minutes): equivalence at init, parameter counts, subsets, trainer, launcher, ...
if RUN_TESTS:
    sh([sys.executable, "-m", "pytest", "-q", "-x", "-p", "no:cacheprovider"])
else:
    print("skipped (RUN_TESTS is off)")

........................................................................ [ 48%]
........................................................................ [ 97%]
....                                                                     [100%]
148 passed in 45.61s


In [6]:
# 6 · One tiny training run: catches setup problems (dataset download, paths) before a long launch
if PLAN != "smoke":
    scratch = WORK / "smoke_scratch"
    shutil.rmtree(scratch, ignore_errors=True)
    sh([sys.executable, "scripts/train.py", "--config", "configs/experiments/smoke.yaml",
        "--set", f"paths.results_dir={scratch}/raw", "--set", f"paths.runs_dir={scratch}/runs",
        "--set", f"paths.data_dir={DATA_DIR}"])
else:
    print("not needed: the smoke plan below runs the same thing for every model x connection")

not needed: the smoke plan below runs the same thing for every model x connection


In [7]:
# 7 · Time one training step of every model x connection in the plan and price the plan in GPU-hours
if RUN_TIMING and PLAN != "smoke":
    models = plan["models"] or ["resnet20", "resnet56", "resnet110", "vit_tiny_d6", "vit_tiny_d12", "vit_tiny_d24"]
    conns = plan["connections"] or ["residual", "dhc_n2", "dhc_n4", "mhc_n4"]
    data = [d.replace("cifar100_", "") for d in (plan["data"] or ["cifar100_p100", "cifar100_p10"])]
    sh([sys.executable, "scripts/time_epoch.py", "--models", *models, "--connections", *conns])
    csv_path = sorted(glob.glob(str(CODE_DIR / "results/tables/timing_*.csv")))[-1]
    (BASE / "timing").mkdir(exist_ok=True)
    shutil.copy(csv_path, BASE / "timing" / os.path.basename(csv_path))
    sh([sys.executable, "scripts/budget.py", csv_path, "--epochs", plan["epochs"], "--seeds", len(plan["seeds"]),
        "--models", *models, "--connections", *conns, "--data", *data, "--steps", P10_STEPS])
    print("\nTiming CSV also saved to", BASE / "timing")
else:
    print("skipped")

skipped


In [8]:
# 8 · Run. Safe to run again after a disconnect: finished runs are skipped, interrupted ones resume.
if LAUNCH:
    t0 = time.time()
    sh([sys.executable, "scripts/run_matrix.py", "--tier", plan["tier"], *GRID, *SETS])
    print(f"finished in {(time.time() - t0) / 3600:.2f} h")
else:
    print("skipped (LAUNCH is off)")

24 runs, 24 already complete
finished in 0.00 h


In [9]:
# 9 · What is finished (a progress view only: the study's numbers come from the run folders, not from this table)
import pandas as pd

rows = [json.loads(f.read_text(encoding="utf-8")) for f in sorted(RAW.glob("*/final.json"))]
if rows:
    df = pd.DataFrame(rows)
    df["hours"] = df["train_seconds"] / 3600
    pd.set_option("display.max_rows", 300)
    display(df[["run_id", "epochs_done", "final_acc", "best_acc", "diverged", "hours", "peak_mem_mb"]].round(4))
    print(f"{len(df)} runs complete | {int(df['diverged'].sum())} diverged | {df['hours'].sum():.1f} GPU-hours of training")
else:
    print("no finished runs yet")
failed = RAW / "_failures.txt"
if failed.exists():
    print("FAILURES (see each run's train.log):\n" + failed.read_text(encoding="utf-8")[-3000:])

,run_id,epochs_done,final_acc,best_acc,diverged,hours,peak_mem_mb
0,resnet110_dhc_n2_cifar100p10_s0,2,0.0120,0.0121,False,0.0019,1849.0444
1,resnet110_dhc_n4_cifar100p10_s0,2,0.0118,0.0123,False,0.0022,2377.8623
2,resnet110_mhc_n4_cifar100p10_s0,2,0.0144,0.0144,False,0.0036,2401.4185
3,resnet110_residual_n1_cifar100p10_s0,2,0.0129,0.0129,False,0.0012,1071.1030
4,resnet20_dhc_n2_cifar100p10_s0,2,0.0215,0.0215,False,0.0008,362.3511
5,resnet20_dhc_n4_cifar100p10_s0,2,0.0199,0.0199,False,0.0007,463.7476
6,resnet20_mhc_n4_cifar100p10_s0,2,0.0193,0.0193,False,0.0011,467.5293
7,resnet20_residual_n1_cifar100p10_s0,2,0.0202,0.0202,False,0.0006,348.7495
8,resnet56_dhc_n2_cifar100p10_s0,2,0.0172,0.0172,False,0.0012,956.6206
9,resnet56_dhc_n4_cifar100p10_s0,2,0.0158,0.0158,False,0.0012,1228.6001


24 runs complete | 0 diverged | 0.0 GPU-hours of training


In [10]:
# 10 · Package the results: one zip on Drive, to unzip into the project's results/raw/
if any(RAW.glob("*/final.json")):
    zip_path = shutil.make_archive(str(BASE / f"{RUN_NAME}_raw"), "zip", RAW)
    print(f"{zip_path}  ({os.path.getsize(zip_path) / 2**20:.1f} MB)")
    print("Optional direct download to your computer:")
    print(f"    from google.colab import files; files.download({zip_path!r})")
else:
    print("nothing to package yet")

/content/drive/MyDrive/nn09/smoke_raw.zip  (7.7 MB)
Optional direct download to your computer:
    from google.colab import files; files.download('/content/drive/MyDrive/nn09/smoke_raw.zip')


## Bring it back into the project
1. **Executed notebook:** *File ▸ Download ▸ Download .ipynb* and replace `notebooks/nn09_colab.ipynb`.
2. **Results:** unzip `MyDrive/<DRIVE_FOLDER>/<plan>_raw.zip` into `results/raw/` (one folder per run). Keep one plan and epoch
   count per `results/raw/`: the analysis scripts assume all runs share their settings.
3. **Timing table (optional):** `MyDrive/<DRIVE_FOLDER>/timing/timing_<gpu>.csv` goes into `results/tables/`; it lets the project
   lead pick epochs and seeds with `python scripts/budget.py`.

If anything fails, the cell output above it says where; the first thing to try is *Run all* again (runs resume).